# 02 - Feature Engineering & Preprocessing

**Purpose:** Transform raw data into model-ready features with best practices.

**Key Improvements:**
- Target clipping at 99th percentile (prevents outlier dominance in RMSE)
- Outlier-robust rent percentiles (not absolute values)
- Categorical feature tracking for CatBoost
- Fold-aware target encoding (prevents leakage)
- Non-linear age features (yearbuilt is #1 importance)
- Pre/Post COVID interaction features

**Output:**
- `data/processed/train_clean.csv`
- `data/processed/test_clean.csv`
- `data/processed/categorical_features.json`

In [ ]:
# Environment detection
import os
import sys

try:
    import google.colab
    USE_COLAB = True
except ImportError:
    USE_COLAB = False

PROJECT_NAME = '03_Rice_Datathon_Colab'
print(f"Environment: {'Google Colab' if USE_COLAB else 'Local'}")

In [ ]:
# Set up project paths
if USE_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = f'/content/drive/MyDrive/{PROJECT_NAME}'
else:
    PROJECT_ROOT = os.path.dirname(os.getcwd())

sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')

RAW_DIR = Path(PROJECT_ROOT) / 'data' / 'raw'
PROCESSED_DIR = Path(PROJECT_ROOT) / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'target'
SEED = 42
N_FOLDS = 5

---
## Configuration

In [ ]:
# =============================================================================
# CLEANING CONFIG
# =============================================================================

CLEANING_CONFIG = {
    # Columns to drop (identifiers)
    'drop_columns': ['UBID', 'id'],
    
    # Create missing indicators for these columns (missing = informative signal)
    'missing_indicator_cols': ['year_renov', 'food_nearest_high_price_mi'],
    
    # Impute these columns with median
    'impute_median_cols': ['year_renov', 'food_nearest_high_price_mi'],
    
    # Group rare categories (threshold = min frequency)
    'group_rare_cols': {
        'city': 0.01,
        'submrkt_name': 0.01,
    },
    
    # Drop highly correlated features
    'drop_correlated': [
        'ownrent_spread_abs',      # Keep spread_pct (0.97 correlation)
        'ownrent_avg_mortgage',    # Keep avg_rent + spread_pct
        'food_count_fast_food',    # Highly correlated with food_total
        'food_count_fast_casual',
        'food_count_moderate',
        'food_count_cheap',
        'food_count_dessert',
        'food_count_bar_pub',
        'food_count_food_court',
    ],
}

# =============================================================================
# FEATURE ENGINEERING CONFIG
# =============================================================================

FEATURE_CONFIG = {
    'clip_target_percentile': 0.99,  # Clip extreme outliers
    'use_target_encoding': True,
    'target_encoding_smoothing': 20,
}

# Columns for target encoding
TARGET_ENCODING_COLS = [
    'state',
    'mrkt_name', 
    'type_main',
    'msa_ring',
]

# Sun Belt states (high performance)
SUNBELT_STATES = ['AZ', 'SC', 'GA', 'FL', 'NC']

---
## Load Data

In [ ]:
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

train = pd.read_csv(RAW_DIR / 'train.csv')
test = pd.read_csv(RAW_DIR / 'test.csv')

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")
print(f"\nTarget stats: mean={train[TARGET].mean():.4f}, std={train[TARGET].std():.4f}")

---
## Helper Functions

In [ ]:
# =============================================================================
# CLEANING FUNCTIONS
# =============================================================================

def drop_columns(df, columns):
    """Drop columns if they exist."""
    cols_to_drop = [c for c in columns if c in df.columns]
    return df.drop(columns=cols_to_drop)

def create_missing_indicators(df, columns):
    """Create binary indicators for missing values."""
    for col in columns:
        if col in df.columns:
            df[f'{col}_missing'] = df[col].isna().astype(int)
    return df

def impute_median(df, columns, fitted_values=None):
    """Impute missing values with median."""
    if fitted_values is None:
        fitted_values = {}
    for col in columns:
        if col in df.columns:
            if col not in fitted_values:
                fitted_values[col] = df[col].median()
            df[col] = df[col].fillna(fitted_values[col])
    return df, fitted_values

def group_rare_categories(df, config, fitted_mappings=None):
    """Group rare categories into '_OTHER_'."""
    if fitted_mappings is None:
        fitted_mappings = {}
    for col, threshold in config.items():
        if col in df.columns:
            if col not in fitted_mappings:
                value_counts = df[col].value_counts(normalize=True)
                fitted_mappings[col] = value_counts[value_counts < threshold].index.tolist()
            df[col] = df[col].replace(fitted_mappings[col], '_OTHER_')
    return df, fitted_mappings

def fill_remaining_missing(train_df, test_df, target_col):
    """Fill any remaining missing values."""
    fitted = {}
    for col in train_df.columns:
        if col == target_col:
            continue
        if train_df[col].isna().sum() > 0:
            if pd.api.types.is_numeric_dtype(train_df[col]):
                fitted[col] = train_df[col].median()
            else:
                mode_vals = train_df[col].mode()
                fitted[col] = mode_vals.iloc[0] if len(mode_vals) > 0 else 'UNKNOWN'
            train_df[col] = train_df[col].fillna(fitted[col])
    for col, val in fitted.items():
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna(val)
    return train_df, test_df

In [ ]:
# =============================================================================
# FEATURE ENGINEERING FUNCTIONS
# =============================================================================

def fold_aware_target_encoding(train_df, test_df, column, target_col, 
                                smoothing=20, n_folds=5, seed=42):
    """
    Fold-aware target encoding to prevent leakage.
    Uses K-Fold to encode train data, global encoding for test.
    """
    new_col = f"{column}_te"
    global_mean = train_df[target_col].mean()
    
    # Global encoding for test data
    stats = train_df.groupby(column)[target_col].agg(['mean', 'count'])
    stats['encoding'] = (
        stats['count'] * stats['mean'] + smoothing * global_mean
    ) / (stats['count'] + smoothing)
    encoding_map = stats['encoding'].to_dict()
    
    # Fold-wise encoding for train data
    train_df[new_col] = np.nan
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    
    for train_idx, val_idx in kf.split(train_df):
        fold_train = train_df.iloc[train_idx]
        fold_stats = fold_train.groupby(column)[target_col].agg(['mean', 'count'])
        fold_global_mean = fold_train[target_col].mean()
        fold_encoding = (
            fold_stats['count'] * fold_stats['mean'] + smoothing * fold_global_mean
        ) / (fold_stats['count'] + smoothing)
        train_df.loc[train_df.index[val_idx], new_col] = (
            train_df.iloc[val_idx][column].map(fold_encoding)
        )
    
    train_df[new_col] = train_df[new_col].fillna(global_mean)
    test_df[new_col] = test_df[column].map(encoding_map).fillna(global_mean)
    
    return train_df, test_df

---
## Apply Cleaning

In [ ]:
print("=" * 60)
print("CLEANING DATA")
print("=" * 60)

# Drop identifier columns
train = drop_columns(train, CLEANING_CONFIG['drop_columns'])
test = drop_columns(test, CLEANING_CONFIG['drop_columns'])
print(f"After dropping identifiers: Train {train.shape}, Test {test.shape}")

# Create missing indicators
train = create_missing_indicators(train, CLEANING_CONFIG['missing_indicator_cols'])
test = create_missing_indicators(test, CLEANING_CONFIG['missing_indicator_cols'])
print(f"After missing indicators: Train {train.shape}, Test {test.shape}")

# Impute median
train, fitted_medians = impute_median(train, CLEANING_CONFIG['impute_median_cols'])
test, _ = impute_median(test, CLEANING_CONFIG['impute_median_cols'], fitted_medians)

# Group rare categories
train, fitted_rare = group_rare_categories(train, CLEANING_CONFIG['group_rare_cols'])
test, _ = group_rare_categories(test, CLEANING_CONFIG['group_rare_cols'], fitted_rare)

# Drop correlated features
train = drop_columns(train, CLEANING_CONFIG['drop_correlated'])
test = drop_columns(test, CLEANING_CONFIG['drop_correlated'])
print(f"After dropping correlated: Train {train.shape}, Test {test.shape}")

---
## Feature Engineering

In [ ]:
# =============================================================================
# [0] TARGET HANDLING - CLIP OUTLIERS
# =============================================================================
print("\n[0] Target Handling...")

# Check for NaN in target
nan_count = train[TARGET].isna().sum()
if nan_count > 0:
    print(f"  WARNING: {nan_count} NaN values in target - dropping")
    train = train[train[TARGET].notna()].reset_index(drop=True)

# Clip extreme outliers
pct = FEATURE_CONFIG['clip_target_percentile']
target_upper = train[TARGET].quantile(pct)
target_lower = train[TARGET].quantile(1 - pct)

train['target_clipped'] = train[TARGET].clip(target_lower, target_upper)
n_clipped = ((train[TARGET] > target_upper) | (train[TARGET] < target_lower)).sum()

print(f"  Clipped {n_clipped} outliers at {pct*100:.0f}th percentile")
print(f"  Range: [{target_lower:.3f}, {target_upper:.3f}]")

In [ ]:
# =============================================================================
# [1] PROPERTY AGE FEATURES (Most Important!)
# =============================================================================
print("\n[1] Property Age Features...")

# Basic age
train['property_age'] = 2025 - train['yearbuilt']
test['property_age'] = 2025 - test['yearbuilt']

# Clip extreme ages
train['property_age'] = train['property_age'].clip(0, 100)
test['property_age'] = test['property_age'].clip(0, 100)

# Vintage categories
def get_vintage(year):
    if year < 1980: return 'pre_1980'
    elif year < 1990: return 'eighties'
    elif year < 2000: return 'nineties'
    elif year < 2010: return 'two_thousands'
    else: return 'modern'

train['vintage_category'] = train['yearbuilt'].apply(get_vintage)
test['vintage_category'] = test['yearbuilt'].apply(get_vintage)

# Polynomial age terms
train['age_squared'] = train['property_age'] ** 2
test['age_squared'] = test['property_age'] ** 2

train['age_log'] = np.log1p(train['property_age'])
test['age_log'] = np.log1p(test['property_age'])

print("  Added: property_age, vintage_category, age_squared, age_log")

In [ ]:
# =============================================================================
# [2] RENOVATION SIGNALS
# =============================================================================
print("\n[2] Renovation Features...")

# Never renovated indicator
if 'year_renov_missing' in train.columns:
    train['never_renovated'] = train['year_renov_missing'].astype(int)
    test['never_renovated'] = test['year_renov_missing'].astype(int)
else:
    train['never_renovated'] = (train['year_renov'] == train['yearbuilt']).astype(int)
    test['never_renovated'] = (test['year_renov'] == test['yearbuilt']).astype(int)

# Years since renovation
train['years_since_renov'] = np.where(
    train['never_renovated'] == 0,
    2025 - train['year_renov'].fillna(train['yearbuilt']),
    train['property_age']  # If never renovated, use property age
)
test['years_since_renov'] = np.where(
    test['never_renovated'] == 0,
    2025 - test['year_renov'].fillna(test['yearbuilt']),
    test['property_age']
)

# Risk signals
train['old_never_renovated'] = ((train['property_age'] > 30) & (train['never_renovated'] == 1)).astype(int)
test['old_never_renovated'] = ((test['property_age'] > 30) & (test['never_renovated'] == 1)).astype(int)

train['recently_renovated'] = (train['years_since_renov'] <= 5).astype(int)
test['recently_renovated'] = (test['years_since_renov'] <= 5).astype(int)

print("  Added: never_renovated, years_since_renov, old_never_renovated, recently_renovated")

In [ ]:
# =============================================================================
# [3] COVID PERIOD FEATURES
# =============================================================================
print("\n[3] COVID Period Features...")

train['is_post_covid'] = (train['time_window_tag'] == 'post').astype(int)
test['is_post_covid'] = (test['time_window_tag'] == 'post').astype(int)

train['is_pre_covid'] = (train['time_window_tag'] == 'pre').astype(int)
test['is_pre_covid'] = (test['time_window_tag'] == 'pre').astype(int)

print("  Added: is_post_covid, is_pre_covid")

In [ ]:
# =============================================================================
# [4] PROPERTY CLASS FEATURES
# =============================================================================
print("\n[4] Property Class Features...")

train['is_class_d'] = (train['type_main'] == 'D').astype(int)
test['is_class_d'] = (test['type_main'] == 'D').astype(int)

train['is_class_a'] = (train['type_main'] == 'A').astype(int)
test['is_class_a'] = (test['type_main'] == 'A').astype(int)

# Value tier
def get_value_tier(type_main):
    if type_main in ['D', 'C-', 'C']: return 'value'
    elif type_main in ['B-', 'B', 'B+']: return 'mid'
    else: return 'premium'

train['value_tier'] = train['type_main'].apply(get_value_tier)
test['value_tier'] = test['type_main'].apply(get_value_tier)

print("  Added: is_class_d, is_class_a, value_tier")

In [ ]:
# =============================================================================
# [5] GEOGRAPHIC FEATURES
# =============================================================================
print("\n[5] Geographic Features...")

train['is_sunbelt'] = train['state'].isin(SUNBELT_STATES).astype(int)
test['is_sunbelt'] = test['state'].isin(SUNBELT_STATES).astype(int)

train['is_texas'] = (train['state'] == 'TX').astype(int)
test['is_texas'] = (test['state'] == 'TX').astype(int)

# MSA ring numeric
train['msa_ring_num'] = pd.to_numeric(train['msa_ring'], errors='coerce').fillna(3)
test['msa_ring_num'] = pd.to_numeric(test['msa_ring'], errors='coerce').fillna(3)

train['is_downtown'] = (train['msa_ring_num'] == 1).astype(int)
test['is_downtown'] = (test['msa_ring_num'] == 1).astype(int)

train['is_outer_suburb'] = (train['msa_ring_num'] >= 4).astype(int)
test['is_outer_suburb'] = (test['msa_ring_num'] >= 4).astype(int)

print("  Added: is_sunbelt, is_texas, msa_ring_num, is_downtown, is_outer_suburb")

In [ ]:
# =============================================================================
# [6] DRIVETIME FEATURES
# =============================================================================
print("\n[6] Drivetime Features...")

train['is_drv10'] = (train['trade_area_label'] == 'drv10').astype(int)
test['is_drv10'] = (test['trade_area_label'] == 'drv10').astype(int)

train['is_drv30'] = (train['trade_area_label'] == 'drv30').astype(int)
test['is_drv30'] = (test['trade_area_label'] == 'drv30').astype(int)

# Key combination: Outer suburb + tight amenities
train['suburb_tight_amenities'] = train['is_outer_suburb'] * train['is_drv10']
test['suburb_tight_amenities'] = test['is_outer_suburb'] * test['is_drv10']

print("  Added: is_drv10, is_drv30, suburb_tight_amenities")

In [ ]:
# =============================================================================
# [7] HOUSING ECONOMICS (Outlier-Robust)
# =============================================================================
print("\n[7] Housing Economics Features...")

# Percentile-based rent position (robust to outliers)
if 'ownrent_avg_rent' in train.columns:
    train['rent_percentile'] = train.groupby('mrkt_name')['ownrent_avg_rent'].rank(pct=True)
    
    # For test, compute percentile based on train distribution
    test_rent_pcts = []
    train_rent_by_market = train.groupby('mrkt_name')['ownrent_avg_rent'].apply(list).to_dict()
    
    for _, row in test.iterrows():
        market = row['mrkt_name']
        rent = row['ownrent_avg_rent']
        if market in train_rent_by_market:
            market_rents = train_rent_by_market[market]
            pct = sum(r < rent for r in market_rents) / len(market_rents) if market_rents else 0.5
        else:
            pct = 0.5
        test_rent_pcts.append(pct)
    test['rent_percentile'] = test_rent_pcts
    
    train['high_rent'] = (train['rent_percentile'] >= 0.75).astype(int)
    test['high_rent'] = (test['rent_percentile'] >= 0.75).astype(int)
    
    print("  Added: rent_percentile, high_rent")

In [ ]:
# =============================================================================
# [8] POST-COVID HEALTH FEATURES
# =============================================================================
print("\n[8] Health & Amenity Features...")

if 'aarp_met_health_hospital' in train.columns:
    train['health_hospital_x_post'] = train['aarp_met_health_hospital'] * train['is_post_covid']
    test['health_hospital_x_post'] = test['aarp_met_health_hospital'] * test['is_post_covid']
    
    train['healthcare_access'] = (
        train['aarp_met_health_hospital'] * 0.5 +
        train['aarp_score_health'] * 0.5
    )
    test['healthcare_access'] = (
        test['aarp_met_health_hospital'] * 0.5 +
        test['aarp_score_health'] * 0.5
    )
    print("  Added: health_hospital_x_post, healthcare_access")

In [ ]:
# =============================================================================
# [9] FOOD AMENITY FEATURES
# =============================================================================
print("\n[9] Food Amenity Features...")

if 'food_count_sit_down' in train.columns and 'food_total_food' in train.columns:
    train['dining_quality_score'] = (
        train['food_count_sit_down'].fillna(0) * 2.0 +
        train['food_count_cafe'].fillna(0) * 1.5
    ) / (train['food_total_food'].fillna(1) + 1)
    
    test['dining_quality_score'] = (
        test['food_count_sit_down'].fillna(0) * 2.0 +
        test['food_count_cafe'].fillna(0) * 1.5
    ) / (test['food_total_food'].fillna(1) + 1)
    print("  Added: dining_quality_score")

if 'food_nearest_high_price_mi_missing' in train.columns:
    train['has_highend_dining'] = (~train['food_nearest_high_price_mi_missing'].astype(bool)).astype(int)
    test['has_highend_dining'] = (~test['food_nearest_high_price_mi_missing'].astype(bool)).astype(int)
    print("  Added: has_highend_dining")

In [ ]:
# =============================================================================
# [10] CRITICAL INTERACTIONS
# =============================================================================
print("\n[10] Interaction Features...")

# Age x Class D (two most important features)
train['age_x_class_d'] = train['property_age'] * train['is_class_d']
test['age_x_class_d'] = test['property_age'] * test['is_class_d']

# Age x Post-COVID
train['age_x_post'] = train['property_age'] * train['is_post_covid']
test['age_x_post'] = test['property_age'] * test['is_post_covid']

# Suburb x Post-COVID
train['suburb_tight_x_post'] = train['suburb_tight_amenities'] * train['is_post_covid']
test['suburb_tight_x_post'] = test['suburb_tight_amenities'] * test['is_post_covid']

# Value x Sunbelt
train['value_x_sunbelt'] = train['is_class_d'] * train['is_sunbelt']
test['value_x_sunbelt'] = test['is_class_d'] * test['is_sunbelt']

print("  Added: age_x_class_d, age_x_post, suburb_tight_x_post, value_x_sunbelt")

In [ ]:
# =============================================================================
# [11] TARGET ENCODING (Fold-Aware)
# =============================================================================
print("\n[11] Target Encoding...")

if FEATURE_CONFIG['use_target_encoding']:
    for col in TARGET_ENCODING_COLS:
        if col in train.columns:
            train, test = fold_aware_target_encoding(
                train, test, col, TARGET,
                smoothing=FEATURE_CONFIG['target_encoding_smoothing'],
                n_folds=N_FOLDS,
                seed=SEED
            )
            print(f"  Encoded: {col} -> {col}_te")

In [ ]:
# =============================================================================
# FILL REMAINING MISSING VALUES
# =============================================================================
print("\n[12] Filling Remaining Missing Values...")

train, test = fill_remaining_missing(train, test, TARGET)

print(f"  Train missing: {train.isna().sum().sum()}")
print(f"  Test missing: {test.isna().sum().sum()}")

---
## Save Outputs

In [ ]:
# =============================================================================
# IDENTIFY CATEGORICAL FEATURES
# =============================================================================
print("\n--- Identifying Categorical Features ---")

categorical_features = [
    'time_window_tag', 'trade_area_label',
    'state', 'mrkt_name', 'MARKET', 'city', 'submrkt_name',
    'type_main', 'type_sub', 'msa_ring',
    'vintage_category', 'value_tier',
]

# Filter to only include columns that exist
categorical_features = [c for c in categorical_features if c in train.columns]

print(f"Categorical features: {len(categorical_features)}")
print(f"  {categorical_features}")

In [ ]:
print("\n=" * 60)
print("SAVING OUTPUTS")
print("=" * 60)

# Save processed data
train.to_csv(PROCESSED_DIR / 'train_clean.csv', index=False)
test.to_csv(PROCESSED_DIR / 'test_clean.csv', index=False)

# Save categorical features list
with open(PROCESSED_DIR / 'categorical_features.json', 'w') as f:
    json.dump(categorical_features, f, indent=2)

print(f"\nSaved files:")
print(f"  train_clean.csv: {train.shape}")
print(f"  test_clean.csv: {test.shape}")
print(f"  categorical_features.json: {len(categorical_features)} features")

In [ ]:
print("\n" + "=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"\nTrain samples: {len(train):,}")
print(f"Test samples: {len(test):,}")
print(f"Total features: {train.shape[1]}")
print(f"Categorical: {len(categorical_features)}")
print(f"Numeric: {train.shape[1] - len(categorical_features) - 2}")
print(f"\n>>> Next: Run 03_full_ensemble.ipynb <<<")